# DLAV Project - Phase 1

This notebook is the main entry point for training, validation, and submission generation.

- Edit the main run parameters in the next code cell before running the notebook.
- Each execution saves its artifacts under `outputs/runs/<timestamp>_<run_name>/`.
- By default, inference and `submission_phase1.csv` use the best checkpoint of the run (`model.pth`), selected by validation ADE.


## Edit This Block First

Most users only need to change the values in the next code cell.
The parameters are grouped by environment, experiment identity, optimization, and inference/output behavior.


In [ ]:
from pathlib import Path

# ==================================
# Edit this cell before running
# ==================================
# Primary outputs are saved under outputs/runs/<timestamp>_<run_name>/.

# --- Colab and storage behavior ---
REPO_URL = 'https://github.com/math707/dlav-project.git'
COLAB_PROJECT_DIR = Path('/content/dlav-project')
MOUNT_DRIVE_IN_COLAB = True
COLAB_DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/dlav-project-runs')
DOWNLOAD_DATA_IF_MISSING = True
SYNC_RUN_TO_DRIVE = True

# --- Experiment identity and core training settings ---
RUN_NAME = None  # Use None to derive the run folder name from the configuration below.
MODEL_NAME = 'model_a'  # 'baseline', 'model_a', or 'model_b'
BATCH_SIZE = 32
NUM_EPOCHS = 50
TEST_BATCH_SIZE = 250

# --- Optimizer and scheduler settings ---
LEARNING_RATE_OPTIONS = {
    'default': 1e-3,
    'lower': 3e-4,
}
LEARNING_RATE_NAME = 'default'
LEARNING_RATE = LEARNING_RATE_OPTIONS[LEARNING_RATE_NAME]
WEIGHT_DECAY = 0.0  # Try 1e-4 for light regularization.
BACKBONE_LEARNING_RATE = None  # Optional explicit LR for model_b's pretrained ResNet18 backbone.
BACKBONE_LR_SCALE = None  # Optional multiplier such as 0.1 for model_b backbone fine-tuning.
USE_LR_SCHEDULER = True
SCHEDULER_NAME = 'plateau'
SCHEDULER_METRIC = 'val_ADE'
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 4
SCHEDULER_MIN_LR = 1e-5
EARLY_STOPPING_PATIENCE = None  # Optional, for example 6.
EARLY_STOPPING_MIN_DELTA = 0.0

# --- Inference and output behavior ---
RELOAD_BEST_CHECKPOINT_FOR_INFERENCE = True

# --- Quick-look visualization ---
NUM_PREVIEW_EXAMPLES = 4

# Derived experiment label (usually leave unchanged)
EXPERIMENT_NAME = f"{MODEL_NAME}_{LEARNING_RATE_NAME}"
if USE_LR_SCHEDULER:
    EXPERIMENT_NAME += f'_{SCHEDULER_NAME}'
if WEIGHT_DECAY > 0:
    EXPERIMENT_NAME += f'_wd{WEIGHT_DECAY:g}'


## Environment Setup

The next cell bootstraps the repository in both local execution and Google Colab, optionally mounts Google Drive in Colab, and resolves the project paths used everywhere else in the notebook.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def _is_running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def _find_project_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    return None


def _bootstrap_project_root() -> tuple[Path, bool]:
    in_colab = _is_running_in_colab()
    project_root = _find_project_root(Path.cwd())

    if in_colab:
        if project_root is None:
            git_dir = COLAB_PROJECT_DIR / '.git'
            if git_dir.is_dir():
                print(f'Updating repository in {COLAB_PROJECT_DIR}...')
                subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
                project_root = COLAB_PROJECT_DIR
            elif COLAB_PROJECT_DIR.exists():
                if (COLAB_PROJECT_DIR / 'src').is_dir() and (COLAB_PROJECT_DIR / 'notebooks').is_dir():
                    print(f'Using existing project directory in {COLAB_PROJECT_DIR}...')
                    project_root = COLAB_PROJECT_DIR
                else:
                    raise FileExistsError(f'{COLAB_PROJECT_DIR} exists but is not a recognized project root.')
            else:
                print(f'Cloning repository into {COLAB_PROJECT_DIR}...')
                subprocess.check_call(['git', 'clone', REPO_URL, str(COLAB_PROJECT_DIR)])
                project_root = COLAB_PROJECT_DIR
        elif project_root == COLAB_PROJECT_DIR and (COLAB_PROJECT_DIR / '.git').is_dir():
            print(f'Updating repository in {COLAB_PROJECT_DIR}...')
            subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
    elif project_root is None:
        raise FileNotFoundError('Could not find the project root. Open the notebook from inside the repository.')

    os.chdir(project_root)
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    return project_root.resolve(), in_colab


PROJECT_ROOT, IN_COLAB = _bootstrap_project_root()

from src.project_setup import prepare_project_context

PROJECT = prepare_project_context(
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
    mount_drive_in_colab=MOUNT_DRIVE_IN_COLAB,
    drive_runs_root=COLAB_DRIVE_RUNS_ROOT,
)

PROJECT_ROOT = PROJECT.project_root
NOTEBOOKS_DIR = PROJECT.notebooks_dir
SRC_DIR = PROJECT.src_dir
DATA_DIR = PROJECT.data_dir
TRAIN_DIR = PROJECT.train_dir
VAL_DIR = PROJECT.val_dir
TEST_DIR = PROJECT.test_dir
OUTPUTS_DIR = PROJECT.outputs_dir
RUNS_DIR = PROJECT.runs_dir
CHECKPOINT_DIR = PROJECT.checkpoints_dir
SUBMISSION_DIR = PROJECT.submissions_dir
LEGACY_CHECKPOINT_PATH = PROJECT.legacy_checkpoint_path
LEGACY_SUBMISSION_PATH = PROJECT.legacy_submission_path

print(f'Environment: {"Google Colab" if PROJECT.in_colab else "Local"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Data directory: {DATA_DIR}')
print(f'Run directory root: {RUNS_DIR}')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Submission directory: {SUBMISSION_DIR}')
if PROJECT.in_colab:
    if PROJECT.drive_mounted:
        print(f'Google Drive mounted: yes -> {PROJECT.drive_runs_root}')
    elif MOUNT_DRIVE_IN_COLAB:
        print('Google Drive mounted: no -> mount was requested but is not available yet')
    else:
        print('Google Drive mounted: no -> mount disabled in notebook parameters')
else:
    print('Google Drive mounted: not relevant outside Colab')


## Imports, Runtime Setup, and Data Preparation

Core logic lives in `src/`. This cell imports the reusable helpers, configures the runtime, and ensures the expected dataset folders are available before training starts.


In [ ]:
import os
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data_utils import DATASET_SPECS, ensure_all_datasets, has_pkl_files, sorted_pkl_files
from src.dataset import DrivingDataset
from src.logger import Logger
from src.model import build_model
from src.run_utils import (
    build_initial_run_metrics,
    copy_artifact_to_destination,
    create_run_context,
    save_metrics,
    sync_run_to_drive,
    write_summary,
)
from src.submission import generate_submission
from src.train import train
from src.training_setup import build_optimizer, build_scheduler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 2 if (PROJECT.in_colab or os.name != 'nt') else 0
PIN_MEMORY = DEVICE.type == 'cuda'

if DOWNLOAD_DATA_IF_MISSING:
    ensure_all_datasets(data_dir=DATA_DIR, in_colab=PROJECT.in_colab, project_root=PROJECT_ROOT)
else:
    missing_splits = [
        name
        for name, config in DATASET_SPECS.items()
        if not has_pkl_files(DATA_DIR / config['target_subdir'])
    ]
    if missing_splits:
        missing_str = ", ".join(missing_splits)
        raise FileNotFoundError(
            f"Missing extracted dataset folders for: {missing_str}. "
            "Either enable DOWNLOAD_DATA_IF_MISSING or place the files manually under data/."
        )

print(f'Device: {DEVICE}')
print(f'num_workers: {NUM_WORKERS}')
print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Model: {MODEL_NAME}')
print(f'Batch size: {BATCH_SIZE} | Epochs: {NUM_EPOCHS}')
print(f'Learning rate: {LEARNING_RATE} ({LEARNING_RATE_NAME}) | Weight decay: {WEIGHT_DECAY}')
print(f'Scheduler: {SCHEDULER_NAME if USE_LR_SCHEDULER else "off"}')
print(f'Best checkpoint reloaded for inference: {RELOAD_BEST_CHECKPOINT_FOR_INFERENCE}')
print(f'Sync run folder to Drive when possible: {SYNC_RUN_TO_DRIVE}')


## Run Tracking

Each execution writes its primary artifacts into `outputs/runs/<timestamp>_<run_name>/`. The notebook keeps a structured metrics file, a short text summary, and the main checkpoint/submission artifacts together for easy review.


In [ ]:
RUN_CONTEXT = create_run_context(
    project_root=PROJECT_ROOT,
    in_colab=PROJECT.in_colab,
    run_name=RUN_NAME,
    default_run_name=EXPERIMENT_NAME,
    drive_root=PROJECT.drive_runs_root,
)

RUN_METRICS = build_initial_run_metrics(
    RUN_CONTEXT,
    model_name=MODEL_NAME,
    device=str(DEVICE),
    batch_size=BATCH_SIZE,
    learning_rate_name=LEARNING_RATE_NAME,
    learning_rate_options=LEARNING_RATE_OPTIONS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    scheduler_enabled=USE_LR_SCHEDULER,
    scheduler_name=SCHEDULER_NAME,
    scheduler_metric=SCHEDULER_METRIC,
    num_epochs=NUM_EPOCHS,
    legacy_checkpoint_path=LEGACY_CHECKPOINT_PATH,
    legacy_submission_path=LEGACY_SUBMISSION_PATH,
)

save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Run name: {RUN_CONTEXT.run_name}')
print(f'Run directory: {RUN_CONTEXT.run_dir}')
print(f'Best checkpoint path: {RUN_CONTEXT.checkpoint_path}')
print(f'Submission path: {RUN_CONTEXT.submission_path}')
if RUN_CONTEXT.drive_backup_enabled:
    print(f'Drive backup path: {RUN_CONTEXT.drive_run_dir}')
elif PROJECT.in_colab and SYNC_RUN_TO_DRIVE:
    print('Drive backup path: unavailable -> Google Drive is not mounted')
else:
    print('Drive backup path: disabled for this execution')


## Quick Data Check

Before training, we inspect a few random training examples directly from the raw pickle files.


In [ ]:
train_sample_files = sorted_pkl_files(TRAIN_DIR)
if not train_sample_files:
    raise FileNotFoundError(f'No training files found in {TRAIN_DIR}')

num_examples = min(NUM_PREVIEW_EXAMPLES, len(train_sample_files))
sample_paths = random.sample(train_sample_files, k=num_examples)

samples = []
for sample_path in sample_paths:
    with open(sample_path, 'rb') as file:
        samples.append(pickle.load(file))

fig, axes = plt.subplots(1, num_examples, figsize=(4 * num_examples, 4))
axes = np.atleast_1d(axes)
for axis, sample_path, sample in zip(axes, sample_paths, samples):
    axis.imshow(sample['camera'])
    axis.set_title(sample_path.name)
    axis.axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, num_examples, figsize=(4 * num_examples, 4))
axes = np.atleast_1d(axes)
for axis, sample in zip(axes, samples):
    axis.plot(sample['sdc_history_feature'][:, 0], sample['sdc_history_feature'][:, 1], 'o-', color='gold', label='Past')
    axis.plot(sample['sdc_future_feature'][:, 0], sample['sdc_future_feature'][:, 1], 'o-', color='green', label='Future')
    axis.legend()
    axis.axis('equal')
plt.tight_layout()
plt.show()


## Datasets and Dataloaders

The training, validation, and test splits all live under `data/`. The notebook only needs to resolve the `.pkl` files and build the PyTorch dataloaders.


In [ ]:
train_files = sorted_pkl_files(TRAIN_DIR)
val_files = sorted_pkl_files(VAL_DIR)

if not train_files:
    raise FileNotFoundError(f'No training files found in {TRAIN_DIR}')
if not val_files:
    raise FileNotFoundError(f'No validation files found in {VAL_DIR}')

train_dataset = DrivingDataset(train_files)
val_dataset = DrivingDataset(val_files)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')


## Model and Training

The notebook launches training, but the model code, optimizer setup helpers, scheduler setup helpers, and training loop all live in `src/`.


In [ ]:
model = build_model(MODEL_NAME)
optimizer = build_optimizer(
    model,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    backbone_learning_rate=BACKBONE_LEARNING_RATE,
    backbone_lr_scale=BACKBONE_LR_SCALE,
)
scheduler = build_scheduler(
    optimizer,
    enabled=USE_LR_SCHEDULER,
    name=SCHEDULER_NAME,
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    min_lr=SCHEDULER_MIN_LR,
)
logger = Logger(log_path=RUN_CONTEXT.log_path)

TRAINING_SUMMARY = train(
    model,
    train_loader,
    val_loader,
    optimizer,
    logger,
    num_epochs=NUM_EPOCHS,
    scheduler=scheduler,
    scheduler_metric=SCHEDULER_METRIC,
    best_checkpoint_path=RUN_CONTEXT.checkpoint_path,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)
final_metrics = TRAINING_SUMMARY.get('final', {})
best_metrics = TRAINING_SUMMARY.get('best', {})

RUN_METRICS.update(
    {
        'epochs_completed': TRAINING_SUMMARY.get('epochs_completed'),
        'train_loss_final': final_metrics.get('train_loss'),
        'val_loss_final': final_metrics.get('val_loss'),
        'val_ADE_final': final_metrics.get('val_ADE'),
        'val_FDE_final': final_metrics.get('val_FDE'),
        'best_val_ADE': best_metrics.get('val_ADE'),
        'best_val_ADE_epoch': best_metrics.get('epoch'),
        'best_checkpoint_path': best_metrics.get('checkpoint_path'),
        'checkpoint_path': best_metrics.get('checkpoint_path'),
        'best_val_loss_at_best_ADE': best_metrics.get('val_loss'),
        'best_val_FDE_at_best_ADE': best_metrics.get('val_FDE'),
        'final_learning_rate': TRAINING_SUMMARY.get('final_learning_rate'),
        'epoch_history': TRAINING_SUMMARY.get('history', []),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

if best_metrics:
    print(f"Best val ADE: {best_metrics['val_ADE']:.4f} at epoch {best_metrics['epoch']}")
    print(f"Best checkpoint: {best_metrics.get('checkpoint_path')}")
print(f'Run metrics updated: {RUN_CONTEXT.metrics_path}')
print(f'Run log: {RUN_CONTEXT.log_path}')


## Save And Reload Best Checkpoint

The best checkpoint by validation ADE is saved inside the run folder, copied to the historical `outputs/checkpoints/` location for backward compatibility, and optionally reloaded so downstream validation plots and the submission use the best model rather than the last epoch. We also keep the last-epoch weights separately for debugging and comparisons.


In [ ]:
LAST_CHECKPOINT_PATH = RUN_CONTEXT.run_dir / 'model_last.pth'
torch.save(model.state_dict(), LAST_CHECKPOINT_PATH)
copy_artifact_to_destination(RUN_CONTEXT.checkpoint_path, LEGACY_CHECKPOINT_PATH)

RUN_METRICS['checkpoint_path'] = str(RUN_CONTEXT.checkpoint_path)
RUN_METRICS['best_checkpoint_path'] = str(RUN_CONTEXT.checkpoint_path)
RUN_METRICS['last_checkpoint_path'] = str(LAST_CHECKPOINT_PATH)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

drive_backup_dir = sync_run_to_drive(RUN_CONTEXT) if SYNC_RUN_TO_DRIVE else None
if drive_backup_dir is not None:
    RUN_METRICS['drive_backup_enabled'] = True
    RUN_METRICS['drive_backup_path'] = str(drive_backup_dir)
    save_metrics(RUN_CONTEXT, RUN_METRICS)
    write_summary(RUN_CONTEXT, RUN_METRICS)

if RELOAD_BEST_CHECKPOINT_FOR_INFERENCE:
    best_state_dict = torch.load(RUN_CONTEXT.checkpoint_path, map_location=DEVICE)
    model.load_state_dict(best_state_dict)
    model = model.to(DEVICE)
    RUN_METRICS['inference_checkpoint_path'] = str(RUN_CONTEXT.checkpoint_path)
else:
    RUN_METRICS['inference_checkpoint_path'] = str(LAST_CHECKPOINT_PATH)

save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Best model saved to {RUN_CONTEXT.checkpoint_path}')
print(f'Last-epoch model saved to {LAST_CHECKPOINT_PATH}')
if RELOAD_BEST_CHECKPOINT_FOR_INFERENCE:
    print(f'Best checkpoint reloaded for downstream evaluation: {RUN_CONTEXT.checkpoint_path}')
else:
    print(f'Inference continues with the last-epoch model: {LAST_CHECKPOINT_PATH}')
print(f'Historical checkpoint copy saved to {LEGACY_CHECKPOINT_PATH}')
if drive_backup_dir is not None:
    print(f'Run backup synchronized to {drive_backup_dir}')
elif PROJECT.in_colab and SYNC_RUN_TO_DRIVE:
    print('Google Drive is not mounted, so this run currently remains stored only in the Colab VM.')


## Qualitative Validation Plots

We compare predicted and ground-truth future trajectories on a validation batch using whichever checkpoint is currently loaded into `model`.


In [ ]:
val_batch = next(iter(val_loader))
model = model.to(DEVICE)

camera = val_batch['camera'].to(DEVICE)
history = val_batch['history'].to(DEVICE)
future = val_batch['future'].to(DEVICE)

model.eval()
with torch.no_grad():
    pred_future = model(camera, history)

camera_np = camera.cpu().numpy()
history_np = history.cpu().numpy()
future_np = future.cpu().numpy()
pred_future_np = pred_future.cpu().numpy()

num_examples = min(NUM_PREVIEW_EXAMPLES, len(camera_np))
selected_indices = random.sample(range(len(camera_np)), k=num_examples)

fig, axes = plt.subplots(1, num_examples, figsize=(4 * num_examples, 4))
axes = np.atleast_1d(axes)
for axis, index in zip(axes, selected_indices):
    axis.imshow(camera_np[index].transpose(1, 2, 0) / 255.0)
    axis.axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, num_examples, figsize=(4 * num_examples, 4))
axes = np.atleast_1d(axes)
for axis, index in zip(axes, selected_indices):
    axis.plot(history_np[index, :, 0], history_np[index, :, 1], 'o-', color='gold', label='Past')
    axis.plot(future_np[index, :, 0], future_np[index, :, 1], 'o-', color='green', label='Future')
    axis.plot(pred_future_np[index, :, 0], pred_future_np[index, :, 1], 'o-', color='red', label='Predicted')
    axis.legend()
    axis.axis('equal')
plt.tight_layout()
plt.show()


## Submission Generation

The test set contains no future trajectory labels. We run inference once and save the primary submission into the run folder, with an optional historical copy in `outputs/submissions/`. If the best checkpoint was reloaded above, the submission uses that best checkpoint.


In [ ]:
test_files = sorted_pkl_files(TEST_DIR)
if not test_files:
    raise FileNotFoundError(f'No test files found in {TEST_DIR}')

test_dataset = DrivingDataset(test_files, test=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

submission = generate_submission(
    model=model,
    data_loader=test_loader,
    device=DEVICE,
    output_path=RUN_CONTEXT.submission_path,
    legacy_output_path=LEGACY_SUBMISSION_PATH,
    copy_fn=copy_artifact_to_destination,
)

RUN_METRICS['submission_path'] = str(RUN_CONTEXT.submission_path)
RUN_METRICS['submission_shape'] = list(submission.shape)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

drive_backup_dir = sync_run_to_drive(RUN_CONTEXT) if SYNC_RUN_TO_DRIVE else None
if drive_backup_dir is not None:
    RUN_METRICS['drive_backup_enabled'] = True
    RUN_METRICS['drive_backup_path'] = str(drive_backup_dir)
    save_metrics(RUN_CONTEXT, RUN_METRICS)
    write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Submission saved to {RUN_CONTEXT.submission_path}')
print(f'Historical submission copy saved to {LEGACY_SUBMISSION_PATH}')
if drive_backup_dir is not None:
    print(f'Run backup synchronized to {drive_backup_dir}')
elif PROJECT.in_colab and SYNC_RUN_TO_DRIVE:
    print('Google Drive is not mounted, so this run currently remains stored only in the Colab VM.')
print(f'Submission shape: {submission.shape}')
